In [16]:
import numpy as np
import pandas as pd 

from scipy import stats

In [4]:
path="/home/dexter/Documents/GitHub/3D-textures/assets/experimental_corrected_results.csv"
df = pd.read_csv(path)

In [ ]:
n_raw = 5
n_method = 5



# 2. Propagate Standard Errors
# SE = Standard Deviation / sqrt(n)
se_raw = df["Std_Deviation"] / np.sqrt(n_raw)
se_method = df["Method_std_Error"] / np.sqrt(n_method)

# Combined Standard Error (Quadrature Sum)
df["SE_Corrected"] = np.sqrt(se_raw**2 + se_method**2)

# 3. Calculate Z-score and One-Tailed p-value (Testing if Corrected Distance > 0)
df["Z_Score"] = df["Corrected_Distance"] / df["SE_Corrected"]
# One-tailed test because we are verifying if the distance significantly exceeds 0
df["p_value"] = 1 - stats.norm.cdf(df["Z_Score"])

# 4. Convert meters to micrometers (μm) for readable, justified precision
df["Corrected_Distance_um"] = df["Corrected_Distance"] * 1e6
df["SE_Corrected_um"] = df["SE_Corrected"] * 1e6

# 5. Format string for Table 2 (e.g., "10.51 ± 39.51 μm")
df["Formatted_Result"] = df.apply(
    lambda r: f"{r['Corrected_Distance_um']:.2f} ± {r['SE_Corrected_um']:.2f} μm",
    axis=1,
)

# Display relevant outputs
print(df[["Printer", "Comparison", "Formatted_Result", "Z_Score", "p_value"]])

   Printer Comparison    Formatted_Result   Z_Score   p_value
0        E     1 vs 2    10.51 ± 88.46 μm  0.118774  0.452727
1        E     2 vs 3  233.68 ± 163.07 μm  1.433049  0.075922
2        E     3 vs 4   73.43 ± 109.26 μm  0.672009  0.250789
3        E     4 vs 5  336.78 ± 188.99 μm  1.781934  0.037380
4        E     5 vs 6   80.47 ± 111.78 μm  0.719913  0.235789
..     ...        ...                 ...       ...       ...
76       B     1 vs 2    91.38 ± 89.18 μm  1.024669  0.152760
77       B     2 vs 3    20.67 ± 64.18 μm  0.322018  0.373719
78       B     3 vs 4  117.28 ± 127.20 μm  0.922024  0.178258
79       B     4 vs 5   100.48 ± 81.98 μm  1.225692  0.110157
80       B     5 vs 6   61.81 ± 261.82 μm  0.236062  0.406692

[81 rows x 5 columns]


In [19]:
# 1. Calculate Mean Distance and Grand SE per printer
printer_summary = (
    df.groupby("Printer")
    .agg(
        Mean_Distance_um=("Corrected_Distance", lambda x: x.mean() * 1e6),
        Total_Tests=("p_value", "count"),
        # Count how many of the rows actually achieved statistical significance
        Significant_Tests=("p_value", lambda x: (x < 0.05).sum()),
    )
    .reset_index()
)

# 2. Compute the true Grand SE across the printer groups
sum_sq_se = (
    df.groupby("Printer")["SE_Corrected"]
    .apply(lambda x: np.sum(x**2))
    .reset_index()
)
printer_summary = printer_summary.merge(sum_sq_se, on="Printer")
printer_summary["Grand_SE_um"] = (
    np.sqrt(printer_summary["SE_Corrected"]) / printer_summary["Total_Tests"]
) * 1e6

# 3. Calculate the percentage of tests that broke through the noise floor
printer_summary["Significance_Rate"] = (
    printer_summary["Significant_Tests"] / printer_summary["Total_Tests"]
) * 100

# 4. Clean up formatting
printer_summary["Averaged_Geometry"] = printer_summary.apply(
    lambda r: f"{r['Mean_Distance_um']:.2f} ± {r['Grand_SE_um']:.2f} μm", axis=1
)
printer_summary["Success_Ratio"] = printer_summary.apply(
    lambda r: f"{int(r['Significant_Tests'])} / {int(r['Total_Tests'])} ({r['Significance_Rate']:.1f}%)",
    axis=1,
)

print(printer_summary[["Printer", "Averaged_Geometry", "Success_Ratio"]])

  Printer  Averaged_Geometry  Success_Ratio
0       B   56.48 ± 17.42 μm  0 / 30 (0.0%)
1       E  200.61 ± 45.45 μm  1 / 28 (3.6%)
2       R   48.60 ± 26.50 μm  1 / 23 (4.3%)
